# Stage 1 — Cross-sensor visibility between Carbon Mapper/Tanager CH₄ plumes and Sentinel-5P XCH₄

This notebook is the cleaned final workflow for **Stage 1** of the thesis.

It replaces the separate development notebooks/scripts:

- `plumes_scoring.ipynb`
- `plumes_clusters.ipynb`
- `CH4_gee.ipynb`
- the Earth Engine Code Editor JavaScript snippet for ±5 day S5P visualization

The purpose of this stage is not to prove that Sentinel-5P detects each individual Tanager plume.  
Instead, it builds a transparent workflow to:

1. extract Tanager CH₄ plume events from Carbon Mapper output,
2. compute plume-mask statistics (`n_valid`, `mean_ch4`),
3. rank plumes using the normalized weighted score used in the abstract,
4. select the top-ranked seed plumes,
5. associate nearby lower-ranked plumes in a ±120 h and 50 km window,
6. query Sentinel-5P/TROPOMI CH₄ around the top seed plumes,
7. generate the plots and maps used for the Stage 1 thesis/abstract figures.

**Important scientific interpretation:** Sentinel-5P has coarse pixels compared with Tanager.  
The S5P part should be described as *coarse-scale temporal/contextual co-visibility*, not direct validation of individual high-resolution plumes.

## 0. Expected input and output

### Required input CSV

This notebook expects a Carbon Mapper plume CSV exported from Stage 0 or from the Carbon Mapper API/dashboard.

Minimum required columns:

```text
plume_latitude
plume_longitude
datetime
gas
platform
plume_tif
con_tif
```

Optional but useful columns:

```text
plume_id
emission_auto
ipcc_sector
instrument
source_name
```

### Main outputs

The notebook writes:

```text
stage1_outputs/
├── tanager_ch4_cleaned.csv
├── tanager_with_nvalid_meanch4.csv
├── scored_weighted_sum_normalized.csv
├── top_20_plumes_WNS.csv
├── plumes_with_time_slots.csv
├── S5P_daily_timeseries_top20.csv
├── figures/
└── maps/
```

The naming `top_20_plumes_WNS.csv` is preserved because it was used in the previous Stage 1 scripts.

In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================
# In Colab, uncomment and run this cell once.
# Locally, install these in your environment instead.

!pip -q install pandas numpy matplotlib scikit-learn folium rasterio earthengine-api geemap tqdm

In [ ]:
# ============================================================
# 2. Configuration
# ============================================================
from pathlib import Path

# ---- input from Stage 0 / Carbon Mapper dashboard ----
# Change this path to your exported Carbon Mapper CSV.
RAW_CM_CSV = Path("/content/plumes_25p87_71p41_-178p19_-71p23_20250101_20251006.csv")

# ---- output folder ----
OUT_DIR = Path("/content/stage1_outputs")
FIG_DIR = OUT_DIR / "figures"
MAP_DIR = OUT_DIR / "maps"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# ---- ranking parameters ----
TOP_N = 20
W_NVALID = 0.5
W_MEANCH4 = 0.5

# ---- Tanager cluster parameters ----
SPATIAL_RADIUS_KM = 50.0
TEMPORAL_WINDOW_HOURS = 120.0  # +/- 5 days

# ---- Sentinel-5P context parameters ----
EE_PROJECT = "global-booster-421311"  # change if needed
S5P_COLLECTION = "COPERNICUS/S5P/OFFL/L3_CH4"
S5P_BAND = "CH4_column_volume_mixing_ratio_dry_air_bias_corrected"
S5P_FALLBACK_BAND = "CH4_column_volume_mixing_ratio_dry_air"
S5P_BUFFER_M = 50_000
S5P_DAYS_BEFORE_AFTER = 5
S5P_SCALE_M = 7000  # close to native L3 CH4 grid scale

# ---- target plume for detailed figure/map ----
TARGET_RANK = 4

In [ ]:
# ============================================================
# 3. Imports and global helper functions
# ============================================================
import os
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
logging.getLogger("rasterio").setLevel(logging.ERROR)

# GDAL/rasterio settings for remote Carbon Mapper GeoTIFF links
os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "YES")
os.environ.setdefault("CPL_VSIL_CURL_ALLOWED_EXTENSIONS", ".tif,.tiff,.vrt")
os.environ.setdefault("CPL_VSIL_CURL_USE_HEAD", "NO")
os.environ.setdefault("GDAL_NUM_THREADS", "ALL_CPUS")

def minmax_norm(s: pd.Series) -> pd.Series:
    """Min-max normalize a numeric pandas Series to [0, 1]."""
    s = pd.to_numeric(s, errors="coerce")
    s_min, s_max = s.min(skipna=True), s.max(skipna=True)
    if pd.isna(s_min) or pd.isna(s_max) or s_max == s_min:
        return pd.Series(0.0, index=s.index)
    return ((s - s_min) / (s_max - s_min)).fillna(0).clip(0, 1)

def ensure_plume_id(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure a stable plume_id exists."""
    df = df.copy()
    if "plume_id" not in df.columns:
        # Use a deterministic id based on row order if no plume_id is available.
        df["plume_id"] = [f"plume_{i:06d}" for i in range(len(df))]
    df["plume_id"] = df["plume_id"].astype(str)
    return df

def normalize_gas(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize gas labels to CH4 / CO2."""
    df = df.copy()
    if "gas" in df.columns:
        df["gas"] = (
            df["gas"].astype(str)
            .str.upper()
            .str.replace("₄", "4", regex=False)
            .str.replace("₂", "2", regex=False)
            .str.replace("METHANE", "CH4", regex=False)
            .str.replace("CARBON DIOXIDE", "CO2", regex=False)
        )
    else:
        df["gas"] = np.nan
    return df

def save_and_show(fig, path: Path, dpi=300):
    """Save a matplotlib figure and show it."""
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")

## 1. Load and filter Carbon Mapper plumes

This reproduces the first part of `plumes_scoring.ipynb`, but keeps only what is needed for Stage 1:

- use **Tanager**
- use **CH₄**
- parse timestamps
- keep rows with valid coordinates

In [ ]:
# ============================================================
# 4. Load Carbon Mapper CSV and keep Tanager CH4 plumes
# ============================================================
if not RAW_CM_CSV.exists():
    raise FileNotFoundError(
        f"RAW_CM_CSV not found: {RAW_CM_CSV}\n"
        "Update RAW_CM_CSV in the configuration cell."
    )

df_raw = pd.read_csv(RAW_CM_CSV)
df_raw = ensure_plume_id(normalize_gas(df_raw))

required = {"plume_latitude", "plume_longitude", "datetime"}
missing = required - set(df_raw.columns)
if missing:
    raise ValueError(f"Input CSV is missing required columns: {missing}")

df_raw["datetime"] = pd.to_datetime(df_raw["datetime"], errors="coerce", utc=True)
df_raw["plume_latitude"] = pd.to_numeric(df_raw["plume_latitude"], errors="coerce")
df_raw["plume_longitude"] = pd.to_numeric(df_raw["plume_longitude"], errors="coerce")

mask = (
    (df_raw["gas"] == "CH4")
    & df_raw["datetime"].notna()
    & df_raw["plume_latitude"].notna()
    & df_raw["plume_longitude"].notna()
)

if "platform" in df_raw.columns:
    mask &= df_raw["platform"].astype(str).str.upper().eq("TANAGER")

tanager = df_raw.loc[mask].copy().reset_index(drop=True)

out_clean = OUT_DIR / "tanager_ch4_cleaned.csv"
tanager.to_csv(out_clean, index=False)

print(f"Raw rows: {len(df_raw)}")
print(f"Tanager CH4 rows after filtering: {len(tanager)}")
print(f"Saved: {out_clean}")
display(tanager.head())

## 2. Compute plume-mask statistics: `n_valid` and `mean_ch4`

This is the cleaned version of the useful part of `plumes_scoring.ipynb`.

For each Tanager plume:

1. open the visual plume mask GeoTIFF (`plume_tif`);
2. open the concentration GeoTIFF (`con_tif`);
3. resample the concentration image onto the plume-mask grid;
4. keep pixels valid in both;
5. compute:

```text
n_valid  = number of valid concentration pixels inside the plume mask
mean_ch4 = mean concentration value over those valid pixels
```

These two variables are the ranking variables used in the Stage 1 abstract.

In [ ]:
# ============================================================
# 5. Compute n_valid and mean_ch4 on the plume mask
# ============================================================
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.errors import NotGeoreferencedWarning

warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

PLUME_COL = "plume_tif"
CON_COL = "con_tif"

def ropen(path: str):
    """Open local or HTTP(S) raster. Uses /vsicurl fallback for remote GeoTIFFs."""
    if not isinstance(path, str) or not path.strip():
        raise ValueError("Empty raster path")
    path = path.strip()
    try:
        return rasterio.open(path)
    except Exception:
        if path.startswith(("http://", "https://")):
            return rasterio.open("/vsicurl/" + path)
        raise

def compute_mask_stats_for_row(row) -> dict:
    """Return n_valid, mean_ch4, min_ch4, max_ch4 for one Carbon Mapper plume row."""
    plume_path = str(row.get(PLUME_COL, "")).strip()
    con_path = str(row.get(CON_COL, "")).strip()

    out = {
        "n_valid": 0,
        "mean_ch4": np.nan,
        "min_ch4": np.nan,
        "max_ch4": np.nan,
    }

    if not plume_path or not con_path or plume_path.lower() == "nan" or con_path.lower() == "nan":
        return out

    try:
        with ropen(plume_path) as plume, ropen(con_path) as con:
            if plume.count < 3:
                return out

            r = plume.read(1, masked=True)
            g = plume.read(2, masked=True)
            b = plume.read(3, masked=True)

            # Valid plume mask where all RGB bands are valid.
            plume_valid = (~r.mask) & (~g.mask) & (~b.mask)

            # Read concentration on the plume grid.
            with WarpedVRT(
                con,
                crs=plume.crs,
                transform=plume.transform,
                width=plume.width,
                height=plume.height,
                resampling=Resampling.nearest,
            ) as con_on_plume:
                c = con_on_plume.read(1, masked=True)

            valid_all = plume_valid & (~c.mask)
            n_valid = int(valid_all.sum())

            if n_valid > 0:
                vals = c.data[valid_all].astype(float)
                vals = vals[np.isfinite(vals)]
                if vals.size > 0:
                    out.update({
                        "n_valid": int(vals.size),
                        "mean_ch4": float(np.mean(vals)),
                        "min_ch4": float(np.min(vals)),
                        "max_ch4": float(np.max(vals)),
                    })
    except Exception:
        # Keep NaNs on failure.
        pass

    return out

# If n_valid and mean_ch4 are already in the input, you may skip recomputation by setting this to False.
RECOMPUTE_MASK_STATS = True

if (not RECOMPUTE_MASK_STATS) and {"n_valid", "mean_ch4"}.issubset(tanager.columns):
    tanager_stats = tanager.copy()
else:
    if PLUME_COL not in tanager.columns or CON_COL not in tanager.columns:
        raise ValueError(
            f"Missing {PLUME_COL!r} or {CON_COL!r}. "
            "These are required to compute n_valid and mean_ch4."
        )

    rows = []
    for _, row in tqdm(tanager.iterrows(), total=len(tanager), desc="Computing plume-mask stats"):
        rec = row.to_dict()
        rec.update(compute_mask_stats_for_row(row))
        rows.append(rec)

    tanager_stats = pd.DataFrame(rows)

out_stats = OUT_DIR / "tanager_with_nvalid_meanch4.csv"
tanager_stats.to_csv(out_stats, index=False)

print(f"Saved: {out_stats}")
display(tanager_stats[["plume_id", "datetime", "n_valid", "mean_ch4", "min_ch4", "max_ch4"]].head())
print(tanager_stats[["n_valid", "mean_ch4"]].describe())

## 3. Rank plumes and select top seeds

This step keeps the ranking actually used in the Stage 1 abstract:

\[
WS_{norm}=w_1 \cdot nvalid_{norm} + w_2 \cdot meanCH4_{norm}
\]

with equal weights by default:

```text
w1 = 0.5
w2 = 0.5
```

The notebook also saves the full ranked table and the top-20 seed table.

In [ ]:
# ============================================================
# 6. Score and rank plumes with normalized weighted sum
# ============================================================
df = tanager_stats.copy()

df["n_valid"] = pd.to_numeric(df["n_valid"], errors="coerce").fillna(0)
df["mean_ch4"] = pd.to_numeric(df["mean_ch4"], errors="coerce")

df["n_valid_norm"] = minmax_norm(df["n_valid"])
df["mean_ch4_norm"] = minmax_norm(df["mean_ch4"])

df["score_wsum_norm"] = (
    W_NVALID * df["n_valid_norm"]
    + W_MEANCH4 * df["mean_ch4_norm"]
)

df = df.sort_values("score_wsum_norm", ascending=False, kind="mergesort").reset_index(drop=True)
df["rank_wsum_norm"] = np.arange(1, len(df) + 1)

# Preserve previous top_ranked convention: 1 for top seed, 0 for the rest.
df["top_ranked"] = 0
df.loc[df.index[:TOP_N], "top_ranked"] = 1

scored_csv = OUT_DIR / "scored_weighted_sum_normalized.csv"
top_csv = OUT_DIR / f"top_{TOP_N}_plumes_WNS.csv"

df.to_csv(scored_csv, index=False)
top20 = df.head(TOP_N).copy()
top20.to_csv(top_csv, index=False)

print(f"Saved full ranked table: {scored_csv}")
print(f"Saved top-{TOP_N} seed table: {top_csv}")
display(top20[["rank_wsum_norm", "plume_id", "datetime", "n_valid", "mean_ch4", "score_wsum_norm"]].head(TOP_N))

## 4. Associate lower-ranked plumes with top seed plumes

This replaces the multiple clustering cells in `plumes_clusters.ipynb`.

For each lower-ranked plume, the notebook checks whether it falls inside:

```text
distance ≤ 50 km
absolute time difference ≤ 120 h
```

relative to any top seed plume.

If more than one seed satisfies both criteria, the lower-ranked plume is assigned to the seed with:

1. smaller absolute time difference,
2. then smaller distance.

The output table includes time-bin columns used later for temporal plots:

```text
day5_before, day4_before, ..., day1_before, t0, day1, ..., day5
```

In [ ]:
# ============================================================
# 7. Spatio-temporal association of lower-ranked plumes to seed plumes
# ============================================================
from sklearn.neighbors import BallTree

EARTH_RADIUS_KM = 6371.0088

TIME_BINS = [
    "day5_before", "day4_before", "day3_before", "day2_before", "day1_before",
    "t0",
    "day1", "day2", "day3", "day4", "day5",
]

def time_bin_from_delta_hours(delta_h: float, zero_tol_h: float = 1e-6) -> str | None:
    """Assign a delta time in hours to one of the +/- 5 day bins."""
    if pd.isna(delta_h):
        return None

    if abs(delta_h) <= zero_tol_h:
        return "t0"

    # After t0: (0,24] -> day1, (24,48] -> day2, ..., (96,120] -> day5
    if delta_h > 0:
        day = int(np.ceil(delta_h / 24.0))
        if 1 <= day <= 5:
            return f"day{day}"
        return None

    # Before t0: [-24,0) -> day1_before, [-48,-24) -> day2_before, ...
    day = int(np.ceil(abs(delta_h) / 24.0))
    if 1 <= day <= 5:
        return f"day{day}_before"
    return None

def associate_to_seed_plumes(ranked_df: pd.DataFrame) -> pd.DataFrame:
    """Attach each non-top plume to the nearest top seed satisfying space-time thresholds."""
    d = ranked_df.copy()
    d["datetime"] = pd.to_datetime(d["datetime"], errors="coerce", utc=True)

    # Initialize association columns.
    d["associated_top_rank"] = np.nan
    d["associated_top_plume_id"] = pd.NA
    d["distance_to_top_km"] = np.nan
    d["delta_t_hours"] = np.nan

    for col in TIME_BINS:
        d[col] = 0

    top = d[d["top_ranked"] == 1].copy()
    rest = d[d["top_ranked"] == 0].copy()

    if top.empty:
        raise ValueError("No top-ranked seed plumes found.")

    # Top plumes are associated with themselves at t0.
    for idx, row in top.iterrows():
        d.loc[idx, "associated_top_rank"] = int(row["rank_wsum_norm"])
        d.loc[idx, "associated_top_plume_id"] = row["plume_id"]
        d.loc[idx, "distance_to_top_km"] = 0.0
        d.loc[idx, "delta_t_hours"] = 0.0
        d.loc[idx, "t0"] = 1

    # Spatial candidate search using haversine BallTree.
    top_coords_rad = np.deg2rad(top[["plume_latitude", "plume_longitude"]].astype(float).values)
    rest_coords_rad = np.deg2rad(rest[["plume_latitude", "plume_longitude"]].astype(float).values)
    tree = BallTree(top_coords_rad, metric="haversine")

    candidate_indices = tree.query_radius(
        rest_coords_rad,
        r=SPATIAL_RADIUS_KM / EARTH_RADIUS_KM,
        return_distance=True,
        sort_results=True,
    )

    top_index_list = top.index.to_numpy()

    for rest_pos, rest_idx in enumerate(rest.index):
        candidate_pos = candidate_indices[0][rest_pos]
        candidate_dist_rad = candidate_indices[1][rest_pos]

        if len(candidate_pos) == 0:
            continue

        r = d.loc[rest_idx]
        valid_candidates = []

        for pos, dist_rad in zip(candidate_pos, candidate_dist_rad):
            top_idx = top_index_list[pos]
            seed = d.loc[top_idx]

            delta_h = (r["datetime"] - seed["datetime"]).total_seconds() / 3600.0
            if abs(delta_h) <= TEMPORAL_WINDOW_HOURS:
                valid_candidates.append({
                    "top_idx": top_idx,
                    "top_rank": int(seed["rank_wsum_norm"]),
                    "top_plume_id": seed["plume_id"],
                    "distance_km": float(dist_rad * EARTH_RADIUS_KM),
                    "delta_h": float(delta_h),
                    "abs_delta_h": abs(float(delta_h)),
                })

        if not valid_candidates:
            continue

        # Choose closest in time first, then space.
        best = sorted(valid_candidates, key=lambda x: (x["abs_delta_h"], x["distance_km"]))[0]
        bin_col = time_bin_from_delta_hours(best["delta_h"])

        d.loc[rest_idx, "associated_top_rank"] = best["top_rank"]
        d.loc[rest_idx, "associated_top_plume_id"] = best["top_plume_id"]
        d.loc[rest_idx, "distance_to_top_km"] = best["distance_km"]
        d.loc[rest_idx, "delta_t_hours"] = best["delta_h"]

        if bin_col is not None:
            d.loc[rest_idx, bin_col] = 1

    return d

plumes_slots = associate_to_seed_plumes(df)

slots_csv = OUT_DIR / "plumes_with_time_slots.csv"
plumes_slots.to_csv(slots_csv, index=False)

# Seed-centered cluster summary excluding the top seed itself.
assoc_counts = (
    plumes_slots[(plumes_slots["top_ranked"] == 0) & plumes_slots["associated_top_rank"].notna()]
    .groupby("associated_top_rank")
    .size()
    .reindex(range(1, TOP_N + 1), fill_value=0)
    .astype(int)
)

print(f"Saved: {slots_csv}")
print(f"Total associated lower-ranked plumes: {assoc_counts.sum()}")
display(assoc_counts.rename("associated_lower_ranked_plumes").to_frame().T)
display(plumes_slots.head())

## 5. Local quick-look maps and Tanager-only cluster plots

These figures correspond to the Carbon Mapper/Tanager side of Stage 1:

- top seed overview map,
- number of lower-ranked plumes associated with each seed,
- optional rank-level map,
- Tanager CH₄ proxy per time bin.

The Tanager CH₄ proxy used here is:

\[
E_{proxy} = meanCH4 \times nvalid
\]

It is not a formal emission-rate estimate; it is a plume-mask concentration proxy used for visual comparison with S5P time series.

In [ ]:
# ============================================================
# 8. Figure: top seed map and associated plume counts
# ============================================================
import folium
from folium.plugins import MarkerCluster
from IPython.display import display

def make_top_seed_overview_map(plumes_df: pd.DataFrame, output_html: Path | None = None):
    top = plumes_df[plumes_df["top_ranked"] == 1].copy()
    assoc = plumes_df[(plumes_df["top_ranked"] == 0) & (plumes_df["associated_top_rank"].notna())].copy()

    lat0 = float(top["plume_latitude"].mean())
    lon0 = float(top["plume_longitude"].mean())

    m = folium.Map(location=[lat0, lon0], zoom_start=4, tiles="cartodbpositron")

    # Top seeds
    fg_top = folium.FeatureGroup(name=f"Top-{TOP_N} seed plumes", show=True)
    for _, r in top.iterrows():
        rank = int(r["rank_wsum_norm"])
        folium.CircleMarker(
            [r["plume_latitude"], r["plume_longitude"]],
            radius=7,
            color="red",
            fill=True,
            fill_opacity=0.85,
            tooltip=f"Rank {rank} | {r['plume_id']}",
        ).add_to(fg_top)
        folium.Circle(
            [r["plume_latitude"], r["plume_longitude"]],
            radius=SPATIAL_RADIUS_KM * 1000,
            color="orange",
            fill=False,
            weight=1,
            opacity=0.5,
        ).add_to(fg_top)
    fg_top.add_to(m)

    # Associated non-top plumes
    fg_assoc = folium.FeatureGroup(name="Associated lower-ranked plumes", show=True)
    cluster = MarkerCluster().add_to(fg_assoc)
    for _, r in assoc.iterrows():
        folium.CircleMarker(
            [r["plume_latitude"], r["plume_longitude"]],
            radius=4,
            color="black",
            fill=True,
            fill_opacity=0.7,
            tooltip=f"Assoc. rank {int(r['associated_top_rank'])} | Δt={r['delta_t_hours']:.1f} h | d={r['distance_to_top_km']:.1f} km",
        ).add_to(cluster)
    fg_assoc.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)

    if output_html is not None:
        output_html.parent.mkdir(parents=True, exist_ok=True)
        m.save(str(output_html))
        print(f"Saved map: {output_html}")

    return m

m_top = make_top_seed_overview_map(plumes_slots, MAP_DIR / "stage1_top20_seed_overview.html")
display(m_top)

# Bar plot: number of associated lower-ranked plumes per seed.
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(1, TOP_N + 1)
y = assoc_counts.reindex(x, fill_value=0).values

ax.bar(x, y, color="#f27030", width=0.6)
ax.set_title(f"Top {TOP_N} Tanager plumes matched in space and time with lower-ranked plumes")
ax.set_xlabel("Top plume rank")
ax.set_ylabel(f"Associated plumes within {SPATIAL_RADIUS_KM:.0f} km and ±{TEMPORAL_WINDOW_HOURS/24:.0f} days")
ax.set_xticks(x)
ax.set_ylim(0, max(y.max() + 2, 2))

for xi, yi in zip(x, y):
    if yi > 0:
        ax.text(xi, yi + 0.2, str(int(yi)), ha="center", va="bottom", fontsize=9)

fig.tight_layout()
save_and_show(fig, FIG_DIR / "figure_associated_plumes_by_rank.png")

In [ ]:
# ============================================================
# 9. Optional map for one target rank
# ============================================================
def make_rank_cluster_map(plumes_df: pd.DataFrame, target_rank: int, output_html: Path | None = None):
    grp = plumes_df[plumes_df["associated_top_rank"].astype(float) == float(target_rank)].copy()
    if grp.empty:
        raise ValueError(f"No rows associated with rank {target_rank}")

    seed = grp[grp["top_ranked"] == 1].iloc[0]
    lat0, lon0 = float(seed["plume_latitude"]), float(seed["plume_longitude"])

    m = folium.Map(location=[lat0, lon0], zoom_start=7, tiles="cartodbpositron")

    folium.Circle(
        [lat0, lon0],
        radius=SPATIAL_RADIUS_KM * 1000,
        color="cyan",
        fill=True,
        fill_opacity=0.08,
        weight=2,
        tooltip=f"{SPATIAL_RADIUS_KM:.0f} km buffer",
    ).add_to(m)

    # Associated lower-ranked plumes
    assoc = grp[grp["top_ranked"] == 0].copy()
    for _, r in assoc.iterrows():
        folium.CircleMarker(
            [r["plume_latitude"], r["plume_longitude"]],
            radius=5,
            color="black",
            fill=True,
            fill_opacity=0.9,
            tooltip=f"lower-ranked plume | Δt={r['delta_t_hours']:.1f}h | d={r['distance_to_top_km']:.1f}km",
        ).add_to(m)

    # Top seed last, so it stays visible
    folium.CircleMarker(
        [lat0, lon0],
        radius=8,
        color="red",
        fill=True,
        fill_opacity=0.95,
        tooltip=f"TOP rank {target_rank} | {seed['plume_id']}",
    ).add_to(m)

    if output_html is not None:
        output_html.parent.mkdir(parents=True, exist_ok=True)
        m.save(str(output_html))
        print(f"Saved map: {output_html}")

    return m

m_rank = make_rank_cluster_map(plumes_slots, TARGET_RANK, MAP_DIR / f"stage1_rank_{TARGET_RANK}_cluster.html")
display(m_rank)

In [ ]:
# ============================================================
# 10. Tanager CH4 proxy per time bin for the target rank
# ============================================================
bin_offsets = [-5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5]
bin_labels = [
    "t0-5 days", "t0-4 days", "t0-3 days", "t0-2 days", "t0-1 day",
    "t0", "t0+1 day", "t0+2 days", "t0+3 days", "t0+4 days", "t0+5 days"
]

timebin_to_offset = {
    "day5_before": -5,
    "day4_before": -4,
    "day3_before": -3,
    "day2_before": -2,
    "day1_before": -1,
    "t0": 0,
    "day1": 1,
    "day2": 2,
    "day3": 3,
    "day4": 4,
    "day5": 5,
}

def tanager_proxy_by_rank(plumes_df: pd.DataFrame, target_rank: int, include_top_seed: bool = False):
    d = plumes_df.copy()
    d["tanager_ch4_proxy"] = pd.to_numeric(d["mean_ch4"], errors="coerce").fillna(0) * pd.to_numeric(d["n_valid"], errors="coerce").fillna(0)

    if include_top_seed:
        sub = d[d["associated_top_rank"].astype(float) == float(target_rank)].copy()
    else:
        sub = d[(d["top_ranked"] == 0) & (d["associated_top_rank"].astype(float) == float(target_rank))].copy()

    totals = {off: 0.0 for off in bin_offsets}

    for col, off in timebin_to_offset.items():
        if col in sub.columns:
            totals[off] += sub.loc[sub[col] == 1, "tanager_ch4_proxy"].sum()

    return np.array([totals[off] for off in bin_offsets], dtype=float)

tanager_y = tanager_proxy_by_rank(plumes_slots, TARGET_RANK, include_top_seed=False)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(bin_offsets))

ax.bar(x, tanager_y, color="orange", width=0.65)
ax.set_xticks(x)
ax.set_xticklabels(bin_labels, rotation=45, ha="right")
ax.set_ylabel("Summed Tanager CH4 proxy = meanCH4 × n_valid")
ax.set_title(f"Tanager associated plumes for rank {TARGET_RANK}")
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
save_and_show(fig, FIG_DIR / f"figure_tanager_proxy_rank_{TARGET_RANK}.png")

## 6. Sentinel-5P daily time-series around top seeds

This section replaces the standalone Earth Engine Code Editor script.

For each top seed plume, the notebook:

1. builds a 50 km buffer around the seed location;
2. loops from `t0 - 5 days` to `t0 + 5 days`;
3. computes the daily mean Sentinel-5P CH₄ image;
4. extracts mean/min/max/valid-pixel-count inside the buffer;
5. saves the table as `S5P_daily_timeseries_top20.csv`.

This reproduces the table used for the S5P blue bars/line in the Stage 1 figures.

In [ ]:
# ============================================================
# 11. Initialize Earth Engine
# ============================================================
import ee

try:
    ee.Initialize(project=EE_PROJECT)
    print(f"Earth Engine initialized with project: {EE_PROJECT}")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print(f"Earth Engine authenticated and initialized with project: {EE_PROJECT}")

In [ ]:
# ============================================================
# 12. Build S5P daily time series for top seed plumes
# ============================================================

def top20_to_ee_fc(top_df: pd.DataFrame) -> ee.FeatureCollection:
    feats = []
    for _, r in top_df.iterrows():
        props = {
            "plume_id": str(r["plume_id"]),
            "rank_wsum_norm": int(r["rank_wsum_norm"]),
            "datetime": pd.to_datetime(r["datetime"], utc=True).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "plume_latitude": float(r["plume_latitude"]),
            "plume_longitude": float(r["plume_longitude"]),
        }
        geom = ee.Geometry.Point([props["plume_longitude"], props["plume_latitude"]])
        feats.append(ee.Feature(geom, props))
    return ee.FeatureCollection(feats)

def compute_s5p_daily_timeseries(top_df: pd.DataFrame) -> pd.DataFrame:
    fc = top20_to_ee_fc(top_df)
    offsets = ee.List.sequence(-S5P_DAYS_BEFORE_AFTER, S5P_DAYS_BEFORE_AFTER)

    def per_plume(feature):
        pid = feature.get("plume_id")
        rank = feature.get("rank_wsum_norm")
        lon = ee.Number(feature.get("plume_longitude"))
        lat = ee.Number(feature.get("plume_latitude"))
        t0 = ee.Date.parse("YYYY-MM-dd'T'HH:mm:ss'Z'", feature.get("datetime"))

        pt = ee.Geometry.Point([lon, lat])
        buffer = pt.buffer(S5P_BUFFER_M)

        def per_day(d):
            d = ee.Number(d)
            day = t0.advance(d, "day")
            next_day = day.advance(1, "day")

            daily_img = (
                ee.ImageCollection(S5P_COLLECTION)
                .filterDate(day, next_day)
                .filterBounds(buffer)
                .select(S5P_BAND)
                .mean()
            )

            stats = daily_img.reduceRegion(
                reducer=ee.Reducer.mean()
                    .combine(ee.Reducer.min(), "", True)
                    .combine(ee.Reducer.max(), "", True)
                    .combine(ee.Reducer.count(), "", True),
                geometry=buffer,
                scale=S5P_SCALE_M,
                bestEffort=True,
                maxPixels=1e8,
            )

            return ee.Feature(None, {
                "plume_id": pid,
                "rank_wsum_norm": rank,
                "plume_datetime": feature.get("datetime"),
                "plume_latitude": lat,
                "plume_longitude": lon,
                "date": day.format("YYYY-MM-dd"),
                "offset_day": d,
                "mean_ch4": stats.get(S5P_BAND + "_mean"),
                "min_ch4": stats.get(S5P_BAND + "_min"),
                "max_ch4": stats.get(S5P_BAND + "_max"),
                "valid_count": stats.get(S5P_BAND + "_count"),
            })

        return ee.FeatureCollection(offsets.map(per_day))

    out_fc = ee.FeatureCollection(fc.map(per_plume)).flatten()

    # Only TOP_N * 11 rows, so getInfo is acceptable.
    info = out_fc.getInfo()
    rows = [f["properties"] for f in info["features"]]
    out = pd.DataFrame(rows)

    # Clean numeric fields.
    for c in ["rank_wsum_norm", "offset_day", "mean_ch4", "min_ch4", "max_ch4", "valid_count"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    return out.sort_values(["rank_wsum_norm", "offset_day"]).reset_index(drop=True)

RUN_S5P_QUERY = True

if RUN_S5P_QUERY:
    s5p_ts = compute_s5p_daily_timeseries(top20)
    ts_csv = OUT_DIR / "S5P_daily_timeseries_top20.csv"
    s5p_ts.to_csv(ts_csv, index=False)
    print(f"Saved: {ts_csv}")
    display(s5p_ts.head(20))
else:
    ts_csv = OUT_DIR / "S5P_daily_timeseries_top20.csv"
    s5p_ts = pd.read_csv(ts_csv)
    print(f"Loaded existing: {ts_csv}")

In [ ]:
# ============================================================
# 13. Optional: Earth Engine Drive export template
# ============================================================
# The previous cell pulls the small table directly with getInfo.
# Use this only if you later scale beyond TOP_N=20 and want a Drive export.

# out_fc = ...  # define a larger FeatureCollection if needed
# task = ee.batch.Export.table.toDrive(
#     collection=out_fc,
#     description="S5P_daily_timeseries_top20",
#     fileFormat="CSV"
# )
# task.start()
# print("Started Drive export task.")

## 7. Sentinel-5P maps around a selected rank

This section replaces the interactive Earth Engine Code Editor visualization.

It creates a map with daily Sentinel-5P CH₄ layers for selected offsets, typically:

```text
t0 - 5 days
t0
t0 + 2 days
```

The red point is the top seed plume.  
The black points are the lower-ranked Tanager plumes associated with that seed.

In [ ]:
# ============================================================
# 14. Geemap visualization of S5P daily CH4 layers for one rank
# ============================================================
import geemap

S5P_VIZ = {
    "min": 1750,
    "max": 1950,
    "palette": ["black", "blue", "purple", "cyan", "green", "yellow", "red"],
}

def ee_points_from_df(points_df: pd.DataFrame) -> ee.FeatureCollection:
    feats = []
    for _, r in points_df.iterrows():
        geom = ee.Geometry.Point([float(r["plume_longitude"]), float(r["plume_latitude"])])
        props = {k: (None if pd.isna(v) else v) for k, v in r.items() if k != "geometry"}
        feats.append(ee.Feature(geom, props))
    return ee.FeatureCollection(feats)

def make_s5p_rank_map(target_rank: int, offsets=(-5, 0, 2)):
    seed = top20[top20["rank_wsum_norm"].astype(int) == int(target_rank)].iloc[0]

    lon = float(seed["plume_longitude"])
    lat = float(seed["plume_latitude"])
    t0 = ee.Date(pd.to_datetime(seed["datetime"], utc=True).strftime("%Y-%m-%dT%H:%M:%SZ"))

    pt = ee.Geometry.Point([lon, lat])
    buffer = pt.buffer(S5P_BUFFER_M)

    m = geemap.Map(center=[lat, lon], zoom=7)
    m.add_basemap("CartoDB.Positron")

    # S5P daily layers
    for d in offsets:
        day = t0.advance(int(d), "day")
        next_day = day.advance(1, "day")
        img = (
            ee.ImageCollection(S5P_COLLECTION)
            .filterDate(day, next_day)
            .filterBounds(buffer)
            .select(S5P_BAND)
            .mean()
            .clip(buffer)
        )
        label = f"S5P XCH4 | rank {target_rank} | t0{d:+d} days"
        m.addLayer(img, S5P_VIZ, label, False)

    # Buffer
    m.addLayer(buffer, {"color": "cyan"}, f"{S5P_BUFFER_M/1000:.0f} km buffer", False)

    # Associated Tanager plumes
    assoc = plumes_slots[
        (plumes_slots["top_ranked"] == 0)
        & (plumes_slots["associated_top_rank"].astype(float) == float(target_rank))
    ].copy()

    if not assoc.empty:
        assoc_fc = ee_points_from_df(assoc)
        m.addLayer(assoc_fc.style(**{"color": "black", "pointSize": 5}), {}, "Associated Tanager plumes", True)

    # Top seed point last
    seed_fc = ee.FeatureCollection([ee.Feature(pt, {"rank": int(target_rank), "plume_id": str(seed["plume_id"])})])
    m.addLayer(seed_fc.style(**{"color": "red", "pointSize": 8}), {}, f"Top plume rank {target_rank}", True)

    return m

m_s5p = make_s5p_rank_map(TARGET_RANK, offsets=(-5, 0, 2))
m_s5p

## 8. Final combined figure: S5P XCH₄ vs Tanager CH₄ proxy

This reproduces the logic behind the Stage 1 combined plot:

- left axis: daily mean Sentinel-5P XCH₄ inside the 50 km buffer;
- right axis: summed Tanager plume-mask CH₄ proxy in the same time bins.

The plot is intended as qualitative temporal co-visibility/context, not as quantitative emission validation.

In [ ]:
# ============================================================
# 15. Combined S5P / Tanager plot for one target rank
# ============================================================
def s5p_series_by_rank(s5p_df: pd.DataFrame, target_rank: int) -> np.ndarray:
    sub = s5p_df[s5p_df["rank_wsum_norm"].astype(int) == int(target_rank)].copy()
    y = []
    for off in bin_offsets:
        vals = sub.loc[sub["offset_day"] == off, "mean_ch4"]
        y.append(vals.iloc[0] if not vals.empty else np.nan)
    return np.array(y, dtype=float)

def plot_s5p_tanager_rank(target_rank: int, save=True):
    s5p_y = s5p_series_by_rank(s5p_ts, target_rank)
    tan_y = tanager_proxy_by_rank(plumes_slots, target_rank, include_top_seed=False)

    x = np.arange(len(bin_offsets))
    width = 0.35

    fig, ax1 = plt.subplots(figsize=(9, 4))

    bars_s5p = ax1.bar(
        x - width / 2,
        s5p_y,
        width=width,
        color="steelblue",
        alpha=0.9,
        label="S5P mean XCH4",
    )
    ax1.set_ylabel("S5P XCH4 [ppb]", color="steelblue")
    ax1.tick_params(axis="y", labelcolor="steelblue")
    ax1.set_ylim(1750, 2000)

    ax1.set_xticks(x)
    ax1.set_xticklabels(bin_labels, rotation=45, ha="right", fontsize=8)

    ax2 = ax1.twinx()
    bars_tan = ax2.bar(
        x + width / 2,
        tan_y,
        width=width,
        color="orange",
        alpha=0.75,
        label="Tanager CH4 proxy",
    )
    ax2.set_ylabel("Tanager CH4 proxy = meanCH4 × n_valid", color="orange")
    ax2.tick_params(axis="y", labelcolor="orange")

    ax1.grid(axis="y", alpha=0.3)
    ax1.set_title(f"Tanager plume rank {int(target_rank)}")

    handles = [bars_s5p, bars_tan]
    labels = [h.get_label() for h in handles]
    ax1.legend(handles, labels, loc="upper right")

    fig.tight_layout()
    if save:
        save_and_show(fig, FIG_DIR / f"figure_s5p_tanager_rank_{int(target_rank)}.png")
    else:
        plt.show()

plot_s5p_tanager_rank(TARGET_RANK, save=True)

In [ ]:
# ============================================================
# 16. Optional: generate combined plots for all top ranks
# ============================================================
GENERATE_ALL_RANK_FIGURES = False

if GENERATE_ALL_RANK_FIGURES:
    for r in range(1, TOP_N + 1):
        plot_s5p_tanager_rank(r, save=True)

## 9. What was removed from the development scripts

The final notebook intentionally removes these development-only parts:

1. **Wind-aware flux ranking**  
   This was experimental and is not part of the Stage 1 abstract methodology.

2. **Nearest-pass S5P matcher from `CH4_gee.ipynb`**  
   The Stage 1 PDF/abstract figure uses daily ±5 day S5P context, not only nearest-pass matching.

3. **Multiple duplicate folium map versions**  
   They were replaced by one overview map and one rank-specific cluster map.

4. **Manual Earth Engine Code Editor UI panel**  
   The final notebook provides a Python/Colab Earth Engine equivalent and is easier to store in the thesis GitHub repository.

5. **Hardcoded Earth Engine assets**  
   The workflow now starts from local CSV files generated by Stage 0 / Carbon Mapper download.  
   This makes the notebook more reproducible and less dependent on private Earth Engine assets.

## 10. GitHub checklist

Before uploading this Stage 1 notebook to GitHub:

- clear all notebook outputs if they contain private file paths;
- do not include API tokens;
- include a small `README.md` describing the required input CSV;
- include outputs only if they are small and non-sensitive;
- keep large rasters and generated maps out of Git if possible.

Recommended folder:

```text
stage1_cross_sensor_visibility/
├── stage1_cross_sensor_visibility_final.ipynb
├── README.md
└── example_config.md
```